# Chart & Graph Question Answering



In [ ]:
import os, base64
from PIL import Image
from io import BytesIO

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# ── Configuration ──────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "..."   

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",   
    temperature=0
)

In [ ]:
# ── Helpers ────────────────────────────────────────────────────
def image_to_base64(image_path: str) -> str:
    """Convert any image file to a base64 string."""
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        buf = BytesIO()
        img.save(buf, format="JPEG")
        return base64.b64encode(buf.getvalue()).decode()

def build_image_message(b64: str, question: str) -> HumanMessage:
    """Wrap base64 image + question into a LangChain HumanMessage."""
    return HumanMessage(content=[
        {"type": "image_url",
         "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        {"type": "text", "text": question}
    ])

In [ ]:
# ── Step 1 : Extract chart context (one-shot, no memory needed) ─
EXTRACT_PROMPT = """
You are a data analyst. Carefully examine the chart/graph and extract:
1. Chart type (bar, line, pie, etc.)
2. Title and axis labels
3. All data points / values you can read
4. Notable trends or patterns
Return a structured plain-text summary only.
Be precise. For stacked bars, estimate each segment separately then sum.
Do NOT round aggressively. If unsure, say "approx".
Cross-check: total stacked height must equal the sum of segments.
"""

def extract_chart_context(image_path: str) -> str:
    b64 = image_to_base64(image_path)
    msg = build_image_message(b64, EXTRACT_PROMPT)
    response = llm.invoke([SystemMessage(content="You are a chart reading expert."), msg])
    return response.content

# ── Step 2 : Memory + LCEL Q&A chain ───────────────────────────
memory = ConversationBufferMemory(return_messages=True, memory_key="history")

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data analyst assistant. Use the chart context below to answer questions.\n\n"
     "CHART CONTEXT:\n{chart_context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

qa_chain = (
    RunnablePassthrough.assign(history=RunnableLambda(lambda _: memory.load_memory_variables({})["history"]))
    | qa_prompt
    | llm
    | StrOutputParser()
)

def ask(chart_context: str, question: str) -> str:
    answer = qa_chain.invoke({"chart_context": chart_context, "question": question})
    memory.save_context({"input": question}, {"output": answer})
    return answer

In [ ]:
# ── Step 3 : Summary Report via Output Parser ──────────────────
from langchain_core.output_parsers import StrOutputParser

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a business analyst. Write a concise executive summary report."),
    ("human",
     "Chart context:\n{chart_context}\n\n"
     "Q&A history:\n{qa_history}\n\n"
     "Generate a structured report with: Key Findings, Trends, and Recommendations.")
])

report_chain = report_prompt | llm | StrOutputParser()

def generate_report(chart_context: str) -> str:
    history = memory.load_memory_variables({})["history"]
    qa_text = "\n".join(
        f"{m.type.upper()}: {m.content}" for m in history
    )
    return report_chain.invoke({"chart_context": chart_context, "qa_history": qa_text})

In [ ]:
# ── DEMO ───────────────────────────────────────────────────────
# Replace with your actual chart image path
IMAGE_PATH = "chart.png"

print("🔍 Extracting chart context...")
chart_context = extract_chart_context(IMAGE_PATH)
print(chart_context)

In [ ]:
# Ask questions (memory persists across calls)
questions = [
    "Which month had the highest sales?",
    "What is the overall trend?",
    "Which category performed worst?"
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {ask(chart_context, q)}")

In [ ]:
# Generate final summary report
print("\n📄 SUMMARY REPORT\n" + "="*50)
print(generate_report(chart_context))

In [ ]:

while True:
    q = input("Ask a question (or 'report' / 'exit'): ").strip()
    if q == "exit":
        break
    elif q == "report":
        print(generate_report(chart_context))
    else:
        print(ask(chart_context, q))